In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error

# 1. load the cleaned data
df = pd.read_csv("cleaned_employee_dataset.csv")

x = df.drop('Salary_INR', axis=1)
y = df['Salary_INR']

# 80-20 split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# 2. pick the top 8 features
print("Picking the top 8 features...")
lin_model = LinearRegression()
sfs = SequentialFeatureSelector(lin_model, n_features_to_select=8, direction='forward')

x_train_sel = sfs.fit_transform(x_train, y_train)
x_test_sel = sfs.transform(x_test)

# dictionary to store all our scores for the final table later
model_scores = {}

# 3. run the Phase 1 models
print("\n--- Phase 1: Linear Regression ---")
linear_models = {
    "Linear Reg   ": LinearRegression(),
    "Ridge Reg    ": Ridge(alpha=1.0),
    "Lasso Reg    ": Lasso(alpha=0.5)
}

for name, model in linear_models.items():
    model.fit(x_train_sel, y_train)
    preds = model.predict(x_test_sel)
    
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    # save to dictionary
    model_scores[name] = {'R2': r2, 'RMSE': rmse}
    print(f"{name} -> R2: {r2:.4f} | RMSE: ₹ {rmse:.2f}")

Picking the top 8 features...

--- Phase 1: Linear Regression ---
Linear Reg    -> R2: 0.9597 | RMSE: ₹ 100369.95
Ridge Reg     -> R2: 0.9597 | RMSE: ₹ 100444.92
Lasso Reg     -> R2: 0.9597 | RMSE: ₹ 100370.01


In [2]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

print("--- Phase 2: KNN and DCT ---")

# the new models the teacher asked for
nonlinear_models = {
    "KNN (K=5)    ": KNeighborsRegressor(n_neighbors=5),
    "Decision Tree": DecisionTreeRegressor(random_state=42, max_depth=5)
}

for name, model in nonlinear_models.items():
    model.fit(x_train_sel, y_train)
    preds = model.predict(x_test_sel)
    
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    # save to the same dictionary from cell 1
    model_scores[name] = {'R2': r2, 'RMSE': rmse}
    print(f"{name} -> R2: {r2:.4f} | RMSE: ₹ {rmse:.2f}")

--- Phase 2: KNN and DCT ---
KNN (K=5)     -> R2: 0.9087 | RMSE: ₹ 151151.41
Decision Tree -> R2: 0.9358 | RMSE: ₹ 126783.58


In [3]:
# this cell only prints the final stored results from above
print("========================================")
print("        FINAL MODEL COMPARISON          ")
print("========================================")

for name, metrics in model_scores.items():
    # printing with commas for the rupees
    print(f"{name} -> R2: {metrics['R2']:.4f} | RMSE: ₹ {metrics['RMSE']:,.2f}")

        FINAL MODEL COMPARISON          
Linear Reg    -> R2: 0.9597 | RMSE: ₹ 100,369.95
Ridge Reg     -> R2: 0.9597 | RMSE: ₹ 100,444.92
Lasso Reg     -> R2: 0.9597 | RMSE: ₹ 100,370.01
KNN (K=5)     -> R2: 0.9087 | RMSE: ₹ 151,151.41
Decision Tree -> R2: 0.9358 | RMSE: ₹ 126,783.58
